# Training Notebook

In [1]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack import get_no_mod, LWEDataset, get_filename_from_params
from ml_attack.utils import get_continuous_reduction_default_params, get_default_params, get_percentage_true_b, get_train_default_params, cmod, mod_mult

import numpy as np

from scipy.linalg import circulant
from ml_attack.lwe import neg_circ

from ml_attack.continuous_reduction import ContinuousReduction
from concurrent.futures import ProcessPoolExecutor

from itertools import product

np.set_printoptions(linewidth=np.inf)

## Dataset creation

Training debug:

In [2]:
params = get_default_params()
params.update(get_continuous_reduction_default_params())
params.update(get_train_default_params())
params.update({
    'n': 32,
    'q': 3329,
    'k': 2,
    'eta': 2,
    'secret_type': 'cbd',
    'error_type': 'cbd',

    'num_gen': 4,
    'seed': 42,

    'num_matrices': 6,
    'reduction_max_size': 100,
    'float_type': 'd',
    'matrix_config': 'salsa',
    'interleaved_steps': 0,
    'reduction_samples': 64,
    'reduction_resampling': False,
    'warmup_steps': 0,
    'bkz_block_sizes': "20:40:10",
    
    'penalty': 4,
    'verbose': True,
    'continuous_reduction': False,

    "train_percentages": [0.1, 0.3, 0.6, 1]
})

filename = get_filename_from_params(params)

reload = False
if os.path.exists(filename) and reload:
    print(f"Loading dataset from {filename}")
    dataset = LWEDataset.load_reduced(filename)
    params = dataset.params
    dataset.approximate_b()
else:
    print(f"Generating dataset and saving to {filename}")
    dataset = LWEDataset(params)
    dataset.initialize()
    dataset.attack(
        stop_strategy="tour",
        stop_after=1,
        attack_strategy="tour",
        save_at_the_end=True
    )

Generating dataset and saving to ./data_n_32_k_2_s_cbd_74365.pkl
Attacking 4 matrices using 8 threads.
- Algo: bkz2.0_20 | Updated 100/128 | Mean std_B: 405.97 | Min row norm: 601.46 | Min col norm: 542.99
- Algo: bkz2.0_20 | Updated 100/128 | Mean std_B: 424.32 | Min row norm: 583.02 | Min col norm: 556.75
- Algo: bkz2.0_20 | Updated 100/128 | Mean std_B: 377.33 | Min row norm: 560.50 | Min col norm: 533.04
- Algo: bkz2.0_20 | Updated 100/128 | Mean std_B: 409.78 | Min row norm: 573.67 | Min col norm: 551.75
A_reduced shape: (4, 1, 64, 64)
RA shape: (4, 100, 32, 64)
R shape: (4, 100, 32, 64)
Non-zero indices shape: (4, 100, 32)
Tour 1 | Time: 61.49s | Mean std_B: 518.27 | Reduction Factor: 0.0624 | Prob: 0.9987
[BEST 10% STD] True B is the best candidate: 1280 / 1280 (100.00%)
[BEST 30% STD] True B is the best candidate: 3835 / 3840 (99.87%)
[BEST 60% STD] True B is the best candidate: 7670 / 7680 (99.87%)
Secret found after 95.44 seconds.
Confusion Matrix:
       |  -2.0  -1.0   0.0 

In [18]:
dataset.approximate_b()
get_percentage_true_b(dataset, verbose=True)

True B is the best candidate: 12756 / 12800 (99.66%)


np.float64(0.9965625)

In [19]:
num_gen = dataset.params['num_gen']
n = dataset.params['n']
k = dataset.params['k']
q = dataset.params['q']

In [20]:
dataset.A.shape

(256, 64)

In [21]:
dataset.A

array([[  963.,  1193.,     8., ...,   923.,    34., -1299.],
       [  760.,   963.,  1193., ...,   310.,   923.,    34.],
       [-1069.,   760.,   963., ...,  1332.,   310.,   923.],
       ...,
       [ 1446., -1271.,  -508., ...,  -205., -1118.,    70.],
       [ 1533.,  1446., -1271., ...,  -262.,  -205., -1118.],
       [ 1397.,  1533.,  1446., ...,  1226.,  -262.,  -205.]])

In [22]:
A_to_reduce = np.stack([dataset.A[ind] for ind in dataset.indices])
A_to_reduce[0]

array([[  963.,  1193.,     8., ...,   923.,    34., -1299.],
       [  760.,   963.,  1193., ...,   310.,   923.,    34.],
       [-1069.,   760.,   963., ...,  1332.,   310.,   923.],
       ...,
       [  951., -1459.,  1343., ..., -1621., -1076.,  1364.],
       [-1487.,   951., -1459., ...,  1503., -1621., -1076.],
       [ -556., -1487.,   951., ...,  -947.,  1503., -1621.]])

In [23]:
dataset.R[0][0]

array([[ -6.,   8.,  -8., ...,  14.,   4.,  22.],
       [ 29.,  -6.,   8., ...,   7.,  14.,   4.],
       [ 19.,  29.,  -6., ...,  27.,   7.,  14.],
       ...,
       [ -3.,   5.,   3., ...,  -3.,   7.,  17.],
       [  8.,  -3.,   5., ..., -22.,  -3.,   7.],
       [ -8.,   8.,  -3., ...,  -4., -22.,  -3.]])

In [24]:
mod_mult(dataset.R[0][0], A_to_reduce[0], q)

array([[ -86.,   18.,  -36., ...,  -76.,   -9.,  -35.],
       [  -9.,  -86.,   18., ...,   -2.,  -76.,   -9.],
       [-103.,   -9.,  -86., ...,  119.,   -2.,  -76.],
       ...,
       [ -21., -146.,    6., ...,   68.,  145., -121.],
       [  36.,  -21., -146., ...,   35.,   68.,  145.],
       [ -18.,   36.,  -21., ...,    9.,   35.,   68.]])

In [25]:
item = 3
parts = np.split(dataset.R[item][0], k)
parts = np.hstack([neg_circ(part).T for part in parts])

mod_mult(parts, A_to_reduce[item], q)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 64 is different from 16)

In [11]:
R_splitted = np.stack([np.hstack([neg_circ(part).T for part in np.split(dataset.R[i][0], k)]) for i in range(dataset.R.shape[0])])
mod_mult(R_splitted, A_to_reduce, q)

array([[[-26.,  18., -16., -36.,  12.,   2., -11.,  -7., -15., -34.,  -1., -18.,  -2.,  12.,  -3.,   6.],
        [  7., -26.,  18., -16., -36.,  12.,   2., -11.,  -6., -15., -34.,  -1., -18.,  -2.,  12.,  -3.],
        [ 11.,   7., -26.,  18., -16., -36.,  12.,   2.,   3.,  -6., -15., -34.,  -1., -18.,  -2.,  12.],
        [ -2.,  11.,   7., -26.,  18., -16., -36.,  12., -12.,   3.,  -6., -15., -34.,  -1., -18.,  -2.],
        [-12.,  -2.,  11.,   7., -26.,  18., -16., -36.,   2., -12.,   3.,  -6., -15., -34.,  -1., -18.],
        [ 36., -12.,  -2.,  11.,   7., -26.,  18., -16.,  18.,   2., -12.,   3.,  -6., -15., -34.,  -1.],
        [ 16.,  36., -12.,  -2.,  11.,   7., -26.,  18.,   1.,  18.,   2., -12.,   3.,  -6., -15., -34.],
        [-18.,  16.,  36., -12.,  -2.,  11.,   7., -26.,  34.,   1.,  18.,   2., -12.,   3.,  -6., -15.]],

       [[ -7.,  -7.,  16.,  24., -13.,  28.,  26., -27.,  25., -16.,  -3.,   6., -18.,  -6.,   0.,  26.],
        [ 27.,  -7.,  -7.,  16.,  24., -13.,

resampling R for more samples (not working)

In [36]:
dataset.R[0][1][:, n:]

array([[ -3.,   7.,  30., ..., -11.,   5., -10.],
       [ 10.,  -3.,   7., ...,  -3., -11.,   5.],
       [ -5.,  10.,  -3., ..., -16.,  -3., -11.],
       ...,
       [-25.,  10.,   2., ...,  -3.,   7.,  30.],
       [-30., -25.,  10., ...,  10.,  -3.,   7.],
       [ -7., -30., -25., ...,  -5.,  10.,  -3.]])

In [35]:
# Take two different R matrices for the same A_to_reduce matrix (e.g., index 0)
R1 = dataset.R[0][0]
R2 = dataset.R[0][1]

# Split each R into k parts
R1_parts = np.split(R1, k, axis=1)
R2_parts = np.split(R2, k, axis=1)

# Stack the split parts from both R1 and R2 along a new axis
combined_parts = np.hstack([R1_parts[0], R2_parts[1]])
combined_parts

array([[ -6.,   8.,  -8., ..., -11.,   5., -10.],
       [ 29.,  -6.,   8., ...,  -3., -11.,   5.],
       [ 19.,  29.,  -6., ..., -16.,  -3., -11.],
       ...,
       [ -3.,   5.,   3., ...,  -3.,   7.,  30.],
       [  8.,  -3.,   5., ...,  10.,  -3.,   7.],
       [ -8.,   8.,  -3., ...,  -5.,  10.,  -3.]])

In [40]:
mod_mult(combined_parts, A_to_reduce[0], q)

array([[  945., -1359.,  -296., ..., -1394., -1206.,   953.],
       [ -675.,   945., -1359., ..., -1203., -1394., -1206.],
       [ 1052.,  -675.,   945., ...,   591., -1203., -1394.],
       ...,
       [ -524.,  -778.,   334., ...,  1380.,  -339., -1225.],
       [  296.,  -524.,  -778., ...,  -953.,  1380.,  -339.],
       [ 1359.,   296.,  -524., ...,  1206.,  -953.,  1380.]])

From paper "Enhancing MLWE"

In [2]:
import numpy as np
from fractions import Fraction

import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack.utils import mod_mult
from ml_attack.lwe import neg_circ

from fpylll import IntegerMatrix, LLL

np.set_printoptions(linewidth=np.inf)

# Parameters
n = 8
k = 2
q = 251
m = 10                # not divisible by n
h = m // n           # =1
g = m - h*n          # =2

dim_u_full = (h+1)*n # 8
dim_v = k*n          # 4
ambient_dim = dim_u_full + dim_v  # 12

print("Parameters:", dict(n=n, k=k, q=q, m=m, h=h, g=g,
                          ambient_dim=ambient_dim, rank_Lp=m+dim_v))

# Random A_full of shape ( (h+1)n x kn )
rng = np.random.default_rng(42)

# Create A_full using neg_circ for MLWE: each column is neg_circ of a random vector mod q
A_full = np.vstack([np.hstack([neg_circ(rng.integers(0, q, n)) for _ in range(k)]) for _ in range(h+1)])

A_used = A_full[:m, :]

print("\nA_full (8x4):\n", A_full)
print("\nA_used (first 6 rows):\n", A_used)

# --- Build ambient lattice basis B_full ---
I_u = np.eye(dim_u_full, dtype=int)
I_v = np.eye(dim_v, dtype=int)

top = np.hstack([I_u*4, A_full])
bottom = np.hstack([np.zeros((dim_u_full, dim_v), dtype=int), q*I_v])
B_full = np.vstack([top, bottom])

print("\nB_full (12x12) lattice basis:\n", B_full)

# --- Projection Π: zero out u-rows hn+g .. (h+1)n-1 ---
Pi = np.diag([0 if h*n + g <= r < (h+1)*n else 1 for r in range(ambient_dim)])
print("\nProjection Pi:\n", Pi)

# Apply projection
D_scaled = Pi @ B_full
print("\nD_scaled = Pi * B:\n", D_scaled)

D_int = IntegerMatrix.from_matrix(D_scaled.astype(int).tolist())
LLL.reduction(D_int)

D_reduced = np.zeros(D_scaled.shape)
D_int.to_matrix(D_reduced)
print("\nD_reduced (after LLL):\n", D_reduced)

# Remove the (n-g) zero vectors after reduction
# Remove all-zero rows
D_nonzero = D_reduced[~np.all(D_reduced == 0, axis=1)]
# Remove all-zero columns
D_nonzero = D_nonzero[:, ~np.all(D_nonzero == 0, axis=0)]
print("\nD_nonzero (after removing zero rows and columns):\n", D_nonzero)

print("D_nonzero shape:", D_nonzero.shape)
print("D_nonzero rank:", np.linalg.matrix_rank(D_nonzero))
print("m + n*k =", m + n * k)

Parameters: {'n': 8, 'k': 2, 'q': 251, 'm': 10, 'h': 1, 'g': 2, 'ambient_dim': 32, 'rank_Lp': 26}

A_full (8x4):
 [[  22 -175  -21 -215 -108 -110 -164 -194   50 -197 -180 -191 -184 -244 -132  -23]
 [ 194   22 -175  -21 -215 -108 -110 -164   23   50 -197 -180 -191 -184 -244 -132]
 [ 164  194   22 -175  -21 -215 -108 -110  132   23   50 -197 -180 -191 -184 -244]
 [ 110  164  194   22 -175  -21 -215 -108  244  132   23   50 -197 -180 -191 -184]
 [ 108  110  164  194   22 -175  -21 -215  184  244  132   23   50 -197 -180 -191]
 [ 215  108  110  164  194   22 -175  -21  191  184  244  132   23   50 -197 -180]
 [  21  215  108  110  164  194   22 -175  180  191  184  244  132   23   50 -197]
 [ 175   21  215  108  110  164  194   22  197  180  191  184  244  132   23   50]
 [ 128 -232  -45  -93 -125 -113 -210  -32  196  -57 -113 -111 -136 -206 -101 -161]
 [  32  128 -232  -45  -93 -125 -113 -210  161  196  -57 -113 -111 -136 -206 -101]
 [ 210   32  128 -232  -45  -93 -125 -113  101  161  196

In [12]:
R = D_nonzero[:, :m]

In [15]:
R_splitted = np.stack([np.hstack([neg_circ(part).T for part in np.split(R[i][0], k)]) for i in range(R.shape[0])])

IndexError: tuple index out of range

In [19]:
import numpy as np
from typing import List, Tuple, Dict, Any
from tqdm import tqdm

def build_matrices_from_blocks(
    n: int,
    m: int,
    num_blocks: int,
    num_matrices: int,
    seed: int = None,
    verbose: bool = False
) -> Tuple[List[np.ndarray], List[Dict[str, Any]]]:
    """
    Build `num_matrices` matrices each with exactly `m` rows from `blocks`.

    - Always uses a global ordering (queue of block indices).
    - Queue is a random permutation of all blocks; once exhausted, it is refilled with a new random permutation.
    - If num_matrices >= num_blocks: ensures each block is the *first* block of a different matrix.
    - Avoids using the same block twice in a matrix until all blocks have been used there.
    """

    rng = np.random.default_rng(seed)

    matrices_segments = [[] for _ in range(num_matrices)]
    used_blocks = {mi: [] for mi in range(num_matrices)}  # track how many times each block was used

    # --- Coverage offset strategy ---
    coverage_offsets = {bidx: [i * m for i in range((n + m - 1) // m)] for bidx in range(num_blocks)}
    coverage_counters = {bidx: 0 for bidx in range(num_blocks)}
    used_offsets = {bidx: set() for bidx in range(num_blocks)}  # track all chosen offsets

    def choose_offset(bidx: int) -> int:
        """Pick next systematic offset if available, else a fresh random offset (avoid repeats)."""
        cnt = coverage_counters[bidx]
        if cnt < len(coverage_offsets[bidx]):
            # Use systematic coverage offset
            offset = coverage_offsets[bidx][cnt]
            coverage_counters[bidx] += 1
        else:
            # Random but prefer unused offsets
            all_offsets = set(range(n))
            candidates = list(all_offsets - used_offsets[bidx])
            if candidates:
                offset = int(rng.choice(candidates))
            else:
                offset = int(rng.integers(0, n))
        used_offsets[bidx].add(offset)
        return offset
        
    # --- Queue of blocks (reshuffled each epoch) ---
    def next_block(queue: list) -> int:
        if not queue:
            new_epoch = rng.permutation(num_blocks).tolist()
            queue.extend(new_epoch)
        return queue.pop(0)

    # Initialize queue (first epoch)
    queue = rng.permutation(num_blocks).tolist()

    # --- Step 1: Assign first block for coverage ---
    if num_matrices >= num_blocks:
        # one block as first in each distinct matrix
        matrix_indices = rng.choice(num_matrices, size=num_blocks, replace=False).tolist()
        for mi, bidx in zip(matrix_indices, rng.permutation(num_blocks)):
            start = choose_offset(bidx)
            take = min(n, m)
            rotated = ((np.arange(n) - start) % n)  + bidx * n
            matrices_segments[mi].append(rotated[:take])
            used_blocks[mi].append(int(bidx))
    else:
        # distribute blocks round-robin
        for i, bidx in enumerate(rng.permutation(num_blocks)):
            target_mi = i % num_matrices
            start = choose_offset(bidx)
            current_rows = sum(seg.shape[0] for seg in matrices_segments[target_mi])
            take = min(n, m - current_rows)
            if take > 0:
                rotated = ((np.arange(n) - start) % n) + bidx * n
                matrices_segments[target_mi].append(rotated[:take])
                used_blocks[target_mi].append(int(bidx))

    # --- Step 2: Fill each matrix up to m rows ---
    for mi in range(num_matrices):
        current_rows = sum(seg.shape[0] for seg in matrices_segments[mi])
        current_blocks = set(used_blocks[mi])

        while current_rows < m:
            bidx = next_block(queue)
            # If already used in this matrix AND there are still unused blocks left,
            # put it back at the end of the queue and try another, unless this is the last block remaining in the queue
            if bidx in current_blocks and len(current_blocks) < num_blocks:
              # Check if there are any blocks in the queue that have not been used in this matrix
              unused_in_queue = [idx for idx in queue if idx not in current_blocks]
              if unused_in_queue:
                queue.append(bidx)
                continue

            start = choose_offset(bidx)
            need = m - current_rows
            take = min(n, need)
            rotated = ((np.arange(n) - start) % n) + bidx * n

            matrices_segments[mi].append(rotated[:take])
            used_blocks[mi].append(int(bidx))
            current_blocks.add(bidx)
            current_rows += take

    # --- Finalize matrices ---
    matrices = np.stack([np.hstack(segs)[:m] for segs in matrices_segments])

    if verbose:
        # Compute overall coverage as the fraction of unique entries in all matrices
        all_entries = np.concatenate(matrices)
        coverage = len(set(all_entries.flatten())) / (num_blocks * n)
        print(f"Overall coverage: {coverage:.4f}")

        # Compute overall reuse: average number of times each entry appears in all matrices
        unique_entries, counts = np.unique(all_entries, return_counts=True)
        overall_reuse = counts.mean()
        print(f"Overall reuse (mean count per unique entry): {overall_reuse:.4f}")

    return matrices

def uniqueness_stats(matrices):
    ordered = len({M.tobytes() for M in matrices})
    unordered = len({
        tuple(map(tuple, np.atleast_2d(M)[np.lexsort(np.atleast_2d(M).T[::-1])]))
        for M in matrices
    })
    return ordered, unordered

# --- Demo ---
if __name__ == "__main__":
    # toy blocks: 4 blocks, block size n=8
    n = 256
    k = 2
    num_gen = 4
    num_blocks = num_gen * k

    #blocks = [np.array([[b, r] for r in range(n)]) for b in range(num_blocks)]
    m = 1000
    num_matrices = 1000
    min_matrices = (n // (m + 1) + 1) * num_gen * k
    m_min = int(np.ceil(n / (num_matrices // (num_gen * k))))

    print(f"num_blocks={num_blocks}, m={m}, num_matrices={num_matrices}, min_matrices={min_matrices}, m_min={m_min}")

    seen_md = set()
    mats = build_matrices_from_blocks(n, m, num_blocks, num_matrices, seed=3, verbose=True)
    print(mats)

    # --- Stress test: repeat 1000 times to check for infinite loop or stack ---
    for trial in tqdm(range(10000)):
        mats_test = build_matrices_from_blocks(n, m, num_blocks, num_matrices, seed=trial)
        ordered, unordered = uniqueness_stats(mats_test)
        assert ordered == len(mats_test), f"Ordered uniqueness failed at trial {trial}"
        assert unordered == len(mats_test), f"Unordered uniqueness failed at trial {trial}"
    print("Stress test passed: no infinite loop or stack after 1000 runs.")


num_blocks=8, m=1000, num_matrices=1000, min_matrices=8, m_min=3
Overall coverage: 1.0000
Overall reuse (mean count per unique entry): 488.2812
[[1626 1627 1628 ...  410  411  412]
 [1040 1041 1042 ...  236  237  238]
 [ 107  108  109 ...  947  948  949]
 ...
 [ 312  313  314 ... 1473 1474 1475]
 [ 720  721  722 ... 1524 1525 1526]
 [  66   67   68 ... 1841 1842 1843]]


  0%|          | 21/10000 [00:11<1:27:35,  1.90it/s]


KeyboardInterrupt: 